In [5]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [6]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    print(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        print("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    print("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


Querying owner: HTOC Org


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,legacyLink,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,url,indicator
0,6755399460008122,2025-07-02T12:05:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T14:38:27Z,4.0,70.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...","[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,201.182.20.16
1,6755399460008265,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T14:38:25Z,4.0,70.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...","[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,103.124.139.170
2,5629499561376898,2025-07-28T17:34:01Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T14:38:16Z,3.0,80.0,TASK0902923,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,206.168.34.44
3,5629499554313255,2025-06-11T14:46:26Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T14:37:52Z,3.0,73.0,RITM0588707 / TASK0886419,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,194.50.16.252
4,6755399459033733,2025-06-16T17:42:21Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T14:36:54Z,3.0,74.0,RITM0589984,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.220.101.98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1317,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84.0,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 34822, 'name': 'business email comprom...","[{'id': 149988, 'dateAdded': '2023-04-21T14:08...",NaN,NaN,NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,geo.netsupportsoftware.com/location/loca.asp
1319,4778127,2024-07-24T16:46:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-07-25T11:18:30Z,4.0,83.0,VA user reported suspicious External email to ...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 38829, 'name': 'Phishing', 'descriptio...","[{'id': 358785, 'dateAdded': '2024-07-24T16:44...",NaN,NaN,NaN,https://epicphysicianstaffing.com,NaN,epicphysicianstaffing.com,epicphysicianstaffing.com
1320,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83.0,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,https://google,NaN,google,google
1321,4517160,2024-02-05T17:10:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-05T23:23:34Z,3.0,67.0,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35057, 'name': 'FDA Splunk API', 'last...","[{'id': 313616, 'dateAdded': '2024-02-05T16:18...",NaN,NaN,NaN,https://realinvestmentadvice.com/,NaN,realinvestmentadvice.com/,realinvestmentadvice.com/


In [7]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=1)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_15296\3485874090.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250730.csv']
Loaded data from 1 files.


In [8]:
import pandas as pd

df = observed_src.copy()

df = df[df['indicator'].isin(observed_data_df['indicator'])]
# --- FULL TAGS UNNEST + PARTNERS ---

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)

# --- GROUPS VIA EXPLODE + GROUPBY ---

groups_exploded = (
    df[['indicator', 'associatedGroups.data']]
      .explode('associatedGroups.data')
      .dropna(subset=['associatedGroups.data'])
)

group_norm = pd.json_normalize(
    groups_exploded['associatedGroups.data']
).rename(columns={'id':'group_id','name':'group_name'})

groups_exploded = groups_exploded.reset_index(drop=True).join(group_norm)

group_agg = (
    groups_exploded
      .drop_duplicates(subset=['indicator','group_id','group_name'])
      .groupby('indicator', as_index=False)
      .agg({
          'group_id':   lambda ids: list(ids),
          'group_name': lambda names: list(names)
      })
      .rename(columns={'group_id':'group_ids','group_name':'group_names'})
)

# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','hostName','dnsActive','whoisActive','source',
    'address','url'
]

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c not in ['address','ip','source','url','legacyLink']]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)

# apply to all your formerly‑list columns
for col in ['partners','group_ids','group_names'] + tag_fields:
    agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)

display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,tag_description,tag_lastUsed,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners,group_ids,group_names
0,1.204.166.3,5629499541090461,2025-05-14T17:47:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:58Z,3.0,69.0,...,Observations reported by the HHS Ofice of the ...,"2025-07-30T14:36:54Z, 2025-07-30T04:59:37Z, 20...",,,,,,"OS, FDA, CMS, HRSA, CDC",,
1,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-29T07:38:17Z,3.0,71.0,...,Observations reported by the HHS Ofice of the ...,"2025-07-30T14:38:25Z, 2025-07-30T14:36:54Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS",,
2,103.149.86.208,6755399458556969,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:55Z,3.0,73.0,...,Observations reported by the HHS Ofice of the ...,"2025-07-30T14:38:25Z, 2025-07-30T14:36:54Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS",,
3,103.163.80.62,6755399460007700,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-26T07:36:15Z,4.0,70.0,...,,"2025-07-18T13:37:52Z, 2025-07-18T13:37:52Z, 20...",,,,,,,5629499544001057,INDICATOR NOTICE 25178.1 – Iran-Aligned Hackti...
4,103.176.96.138,6755399460007427,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-26T07:36:16Z,4.0,70.0,...,,"2025-07-18T13:37:52Z, 2025-07-18T13:37:52Z, 20...",,,,,,,5629499544001057,INDICATOR NOTICE 25178.1 – Iran-Aligned Hackti...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
867,vtext.com,5182028,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-29T07:38:59Z,3.0,47.0,...,Adversaries may send phishing messages to gain...,"2025-07-30T14:38:25Z, 2025-07-17T13:46:36Z, 20...",T1566,['Initial Access'],1.0,"['Linux', 'macOS', 'Windows', 'SaaS', 'Identit...",6.0,"DHA, NIH, IHS",535913,NIH Phishing Emails 11/27/2024
868,www.deepseek.com,5271608,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-29T07:39:18Z,3.0,84.0,...,Observations reported by the HHS Ofice of the ...,"2025-07-30T14:38:25Z, 2025-01-31T14:04:11Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS","548890, 548570","HTOC-20250131-0903-A, Blocked URLs by NIH for ..."
869,www.shorturl.at/,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90.0,...,,"2024-09-10T10:48:34Z, 2025-07-30T04:59:37Z, 20...",,,,,,"FDA, CDC",455233,"New Recent Login, Review Activity, Twilio Send..."
870,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-30T07:36:04Z,4.0,50.0,...,Observations reported by the HHS Ofice of the ...,"2025-07-30T14:38:25Z, 2025-07-30T14:36:54Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS","332131, 331131","HTOC-20240423-0923-A, VA Hosts Reaching Out to..."


In [9]:
# ──────────────────────────────────────────────────────────────────────────────
# Enrich only final filtered indicators (Shodan & VirusTotalV3) and flatten
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

indicator_values = (
    agg_df["summary"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print(f"Enriching {len(indicator_values)} indicators with Shodan & VirusTotalV3...")

enriched_results = []
failed = []

for value in indicator_values:
    try:
        encoded_value = urllib.parse.quote(value, safe="")
        q = urllib.parse.urlencode({"type": ["Shodan", "VirusTotalV3"]}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        # Build a fresh RequestObject each loop (adjust to your SDK)
        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        resp.raise_for_status()

        data = resp.json()
        data["summary"] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({"summary": value, "error": str(e)})

# If nothing enriched, just carry on
if not enriched_results:
    print("No enrichment data retrieved.")
    recent_tags = agg_df.copy()

else:
    # One row per summary from enrichment payload
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset="summary", keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on="summary", how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[["summary", COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat["summary"] = exploded["summary"].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            """Flatten one level of lists in a sequence, keep dicts intact."""
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            # if everything is hashable & not dict/list, dedupe
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat  # keep as-is when dicts/lists present

        obj_cols = enrich_flat.select_dtypes("object").columns.difference(["summary"])
        num_cols = enrich_flat.columns.difference(obj_cols.union({"summary"}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        # choose your numeric aggregation; 'max' or 'first'
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby("summary", as_index=False)
              .agg(agg_dict)
        )

        # Remove raw list col and merge flattened cols back
        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset="summary")
                       .merge(enrich_wide, on="summary", how="left")
        )

        print(f"Successfully enriched and merged {len(df_enriched)} indicators.")
    else:
        print("Enrichment column not found; nothing to flatten.")

# Optional: report/log failures
if failed:
    print(f"{len(failed)} indicators failed to enrich.")
    # Example: pd.DataFrame(failed).to_json("enrich_failures.json", orient="records")


Enriching 872 indicators with Shodan & VirusTotalV3...


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host arpdcresources.ca cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host bimbinlombardia.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host brppharma.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host calbbs.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress cameron@yourpensionmeeting.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
F

Successfully enriched and merged 838 indicators.
34 indicators failed to enrich.


In [10]:
recent_tags


,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_type,enrich_vulnerabilities,enrich_vtMaliciousCount
0,1.204.166.3,5629499541090461,2025-05-14T17:47:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:58Z,3.0,69.0,...,[China],None,None,[CHINANET-BACKBONE],"[{'transport': 'tcp', 'port': 13001}]",[CHINANET GUIZHOU PROVINCE NETWORK],None,"[Shodan, VirusTotal]",None,4.0
1,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-29T07:38:17Z,3.0,71.0,...,[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],"[Shodan, VirusTotal]",None,7.0
2,103.149.86.208,6755399458556969,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:55Z,3.0,73.0,...,[Viet Nam],None,None,[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],"[{'transport': 'tcp', 'port': 22, 'product': '...",[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],[eol-product],"[Shodan, VirusTotal]",None,5.0
3,103.163.80.62,6755399460007700,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-26T07:36:15Z,4.0,70.0,...,[Indonesia],None,None,[PT Data Arta Sedaya],"[{'transport': 'tcp', 'port': 53, 'data': ' Re...",[PT Data Arta Sedaya],[vpn],"[Shodan, VirusTotal]","[{'name': 'CVE-2018-19052', 'port': 7777, 'des...",1.0
4,103.176.96.138,6755399460007427,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-26T07:36:16Z,4.0,70.0,...,[Indonesia],[g-fiber.co.id],[ip.138-96.g-fiber.co.id],[PT Global Sarana Elektronika],"[{'transport': 'udp', 'port': 161, 'product': ...",[PT Global Sarana Elektronika],None,"[Shodan, VirusTotal]",None,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
867,vtext.com,5182028,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-29T07:38:59Z,3.0,47.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
868,www.deepseek.com,5271608,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-29T07:39:18Z,3.0,84.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
869,www.shorturl.at/,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
870,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-30T07:36:04Z,4.0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')

recent_tags

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,1.204.166.3,2025-05-14T17:47:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:58Z,3.0,69.0,FBI Email Alert May 14 2025 Medium IP IOCs,...,[Guiyang],[China],None,None,[CHINANET-BACKBONE],"[{'transport': 'tcp', 'port': 13001}]",[CHINANET GUIZHOU PROVINCE NETWORK],None,None,4.0
1,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-29T07:38:17Z,3.0,71.0,INC9067814,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,7.0
2,103.149.86.208,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:55Z,3.0,73.0,INC9097535,...,[Hanoi],[Viet Nam],None,None,[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],"[{'transport': 'tcp', 'port': 22, 'product': '...",[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],[eol-product],None,5.0
3,103.163.80.62,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-26T07:36:15Z,4.0,70.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,[Madiun],[Indonesia],None,None,[PT Data Arta Sedaya],"[{'transport': 'tcp', 'port': 53, 'data': ' Re...",[PT Data Arta Sedaya],[vpn],"[{'name': 'CVE-2018-19052', 'port': 7777, 'des...",1.0
4,103.176.96.138,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-26T07:36:16Z,4.0,70.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,[Pamanukan],[Indonesia],[g-fiber.co.id],[ip.138-96.g-fiber.co.id],[PT Global Sarana Elektronika],"[{'transport': 'udp', 'port': 161, 'product': ...",[PT Global Sarana Elektronika],None,None,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
867,vtext.com,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-29T07:38:59Z,3.0,47.0,The correlation search is based on an automate...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
868,www.deepseek.com,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-29T07:39:18Z,3.0,84.0,Executive Summary: \n\nThe following DeepSeek...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
869,www.shorturl.at/,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90.0,ACD R&F processed a malspam campaign with a Ne...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
870,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-30T07:36:04Z,4.0,50.0,The Department of Veterans Affairs (VA) receiv...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
test = recent_tags[recent_tags['indicator'] == '207.167.64.24'] # Malicious

In [41]:
test = recent_tags[recent_tags['indicator'] == '34.145.231.212'] # NonMalicious

In [10]:
recent_tags.loc[recent_tags['indicator'] == '207.167.64.24', ['indicator', 'observations', 'rating', 'confidence']]

,indicator,observations,rating,confidence
430,207.167.64.24,111591,3.0,76.0


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# 1. Define final scoring function (from earlier)
def compute_threat_score(df,
                         lambda_decay=0.1,
                         alpha=0.5,
                         beta=0.5,
                         Cq=50, Cl=150, Cb=0,
                         Co=10,
                         baseline=100,
                         delta=150, theta=1):
    # Map threat_rating/confidence_pct pass-through if numeric; else map manually
    df = df.copy()
    # Quadratic component
    df['S_C'] = Cq * df['Xcrit']**2 + Cl * df['Xcrit'] + Cb
    # Observation component
    df['S_O'] = Co * df['decayed_obs']
    # False-positive penalty
    df['FP_penalty'] = -delta * df['fp_count']**theta
    # Final score
    df['threat_score'] = baseline + df['S_C'] + df['S_O'] + df['FP_penalty']
    return df

# 2. Temporal & Volume features
def compute_temporal_volume_features(df, daily_counts_df, lambda_decay=0.1, today=None):
    if today is None:
        today = pd.Timestamp(datetime.utcnow().date())
    df = df.copy()
    # Normalize datetime columns (drop tz)
    df['dateAdded']   = pd.to_datetime(df['dateAdded']).dt.tz_localize(None).dt.normalize()
    df['lastObserved'] = pd.to_datetime(df['lastObserved']).dt.tz_localize(None).dt.normalize()
    # Raw & recency metrics
    df['raw_obs'] = df['observations']
    df['days_since_first_seen'] = (today - df['dateAdded']).dt.days
    df['days_since_last_seen']  = (today - df['lastObserved']).dt.days
    df['decayed_obs'] = df['raw_obs'] * np.exp(-lambda_decay * df['days_since_first_seen'])
    # Trend & burstiness
    def trend_burst(ind):
        ser = daily_counts_df[daily_counts_df['indicator']==ind].sort_values('date')['count'].values
        if len(ser)>=2:
            x = np.arange(len(ser))
            slope = np.polyfit(x, ser, 1)[0]
            burst = np.std(ser)/np.mean(ser) if np.mean(ser)>0 else 0.0
        else:
            slope, burst = 0.0, 0.0
        return slope, burst
    tb = df['indicator'].apply(trend_burst)
    df[['obs_trend_slope','obs_burstiness']] = pd.DataFrame(tb.tolist(), index=df.index)
    return df

# 3. Source Diversity & Trust
def compute_source_diversity(df, source_sightings_df, trust_map):
    df = df.copy()
    df['num_distinct_sources'] = source_sightings_df.groupby('indicator')['ownerId'].nunique().reindex(df['indicator']).fillna(0).astype(int).values
    agg = source_sightings_df.groupby('indicator')['ownerId'].apply(list)
    sums, maxs, ratios = [], [], []
    for ind, owners in zip(df['indicator'], df['indicator']):
        owners = agg.get(ind, [])
        weights = [trust_map.get(o,5) for o in owners]
        sums.append(sum(weights))
        maxs.append(max(weights) if weights else 0)
        high = sum(w>=7 for w in weights); low = sum(w<7 for w in weights)
        ratios.append(high/(low if low>0 else 1))
    df['sum_source_weight'], df['max_source_weight'], df['high_low_trust_ratio'] = sums, maxs, ratios
    return df

# 4. False-Positive Signals
def compute_fp_signals(df):
    df = df.copy()
    df['fp_count'] = df.get('falsePositiveCount', 0)
    df['fp_rate'] = df['fp_count'] / df['observations'].replace(0,1)
    return df

# 5. Contextual Metadata
def compute_context_metadata(df):
    df = df.copy()
    types = pd.get_dummies(df['type'], prefix='type')
    df = pd.concat([df, types], axis=1)
    df['port_diversity'] = df['enrich_openPorts'].fillna('').apply(lambda p: len(set(str(p).split(','))) if p else 0)
    df['vt_malicious_count'] = df.get('enrich_vtMaliciousCount', 0)
    return df

# 6. Behavioral & Anomaly Signals
def compute_behavioral(df):
    df = df.copy()
    df['port_scan_confidence'] = df['enrich_tags'].str.contains('scanner', case=False).fillna(False).astype(int)
    return df

# 7. Derived Aggregates
def compute_derived(df, incidents_df=None, cal_scores=None):
    df = df.copy()
    df['ever_in_incident'] = False
    if incidents_df is not None:
        df['ever_in_incident'] = df['indicator'].isin(incidents_df['indicator'])
    df['peer_consensus_score'] = df.get('CAL_Score', cal_scores if cal_scores is not None else 0)
    return df

# --- Main pipeline --- #
if __name__ == "__main__":
    # Load example dataset
    df = test
    # Simulate daily counts for last 7 days
    today = pd.Timestamp(datetime.utcnow().date())
    dates = [today - pd.Timedelta(days=i) for i in range(7)]
    daily = []
    for ind in df['indicator']:
        for d in dates:
            daily.append({'indicator':ind, 'date':d, 'count':np.random.poisson(5)})
    daily_df = pd.DataFrame(daily)
    # Simulate source sightings and trust weights
    owners = df['ownerId'].unique()
    trust_map = {o: np.random.choice([1,5,7,10]) for o in owners}
    src_sightings = [{'indicator':ind, 'ownerId':o} for ind in df['indicator'] for o in owners]
    sightings_df = pd.DataFrame(src_sightings)
    # Apply feature computations
    df = compute_temporal_volume_features(df, daily_df)
    df = compute_source_diversity(df, sightings_df, trust_map)
    df = compute_fp_signals(df)
    df = compute_context_metadata(df)
    df = compute_behavioral(df)
    df = compute_derived(df)
    # For demonstration, set Xcrit manually (e.g., blend of rating + confidence mapping)
    # Here we use simple average of mapped rating/confidence
    df['Xcrit'] = ((df['rating'].clip(0,5)/5*2 - 2) + (df['confidence']/100*2 - 2)) / 2
    # Generate final score
    scored = compute_threat_score(df)
    display("Final Threat Scores", scored[['indicator','threat_score']])


C:\Users\jaskew\AppData\Local\Temp\ipykernel_15296\4092281586.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = pd.Timestamp(datetime.utcnow().date())
C:\Users\jaskew\AppData\Local\Temp\ipykernel_15296\4092281586.py:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = pd.Timestamp(datetime.utcnow().date())
C:\Users\jaskew\AppData\Local\Temp\ipykernel_15296\4092281586.py:88: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['port_scan_confiden

'Final Threat Scores'

,indicator,threat_score
462,34.145.231.212,71452.437136


In [43]:
import pandas as pd
import numpy as np

# Assume `scored` DataFrame exists from prior pipeline with 'threat_score'

df = scored.copy()

# 1. Soft-cap transform (Michaelis-Menten style)
# Choose half-saturation constant k (e.g., 2000)
k = 2000
df['threat_score_soft'] = 1000 * df['threat_score'] / (df['threat_score'] + k)

# 2. Hard cap at 1000 just for safety
df['threat_score_soft_capped'] = df['threat_score_soft'].clip(upper=1000)

# 3. Round for readability
df['threat_score_soft_capped'] = df['threat_score_soft_capped'].round().astype(int)

# 4. Re-label bands on the new score
bins = [0, 199, 500, 800, 1000]
labels = ['Low', 'Medium', 'High', 'Critical']
df['threat_label_soft'] = pd.cut(
    df['threat_score_soft_capped'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Display comparison
display("Soft-Capped Threat Score vs Original", df[
    ['indicator', 'threat_score', 'threat_score_soft_capped', 'threat_label_soft']
])


'Soft-Capped Threat Score vs Original'

,indicator,threat_score,threat_score_soft_capped,threat_label_soft
462,34.145.231.212,71452.437136,973,Critical


In [45]:
import pandas as pd

# Assume `scored` DataFrame is available from the prior pipeline run
df = scored.copy()

# Define constants used in scoring
baseline = 100
Cq, Cl, Cb = 50, 150, 0
Co = 10
delta = 150

# Build explanation columns per indicator
explanations = []
for _, row in df.iterrows():
    explanations.append({
        "indicator": row["indicator"],
        "Baseline": baseline,
        "Criticality (S_C)": row["S_C"],
        "Observation Score (S_O)": round(row["S_O"], 2),
        "False-Positive Penalty": row["FP_penalty"],
        "Final Threat Score": round(row["threat_score"], 2)
    })

# Create a DataFrame and display
explanations_df = pd.DataFrame(explanations)
display("Indicator Scoring Breakdown", explanations_df)


'Indicator Scoring Breakdown'

,indicator,Baseline,Criticality (S_C),Observation Score (S_O),False-Positive Penalty,Final Threat Score
0,34.145.231.212,100,-72.895,71425.33,0,71452.44
